In [ ]:
import jax
import jax.numpy as jnp
import diffrax
import plotly.graph_objects as go

In [ ]:
KEY = jax.random.key(seed = 42)
NUMPARAMS = 100
NUMSAMPLES = 20
DIMS = 2

$\def\pder#1#2{\frac{\partial #1}{\partial #2}}$
$\pder{\phi}{t} = k\nabla^2\phi$, где
$\nabla = (\pder{}{x}, \pder{}{y})$

In [ ]:
def f(foo, k):
    hess = jax.hessian(foo)
    def laplacian_hessian(x, p):
        return jnp.trace(hess(x, p))

    def eps(x, p):
        return k*laplacian_hessian(x, p)[jnp.newaxis]
    return eps

def get_dw_dt(foo, k, points_sampler):
    eps = f(foo, k)
    yacobian = jax.jacrev(foo, 1,)
    def dw_dt(x, p):
        e = jnp.atleast_1d(eps(x, p))
        J = yacobian(x, p)
        return jax.tree_util.tree_map(lambda M:jnp.atleast_2d(M).T @ e, J)

    def final_dw_dt(p):
        x = points_sampler()
        q = jax.vmap(lambda x, p:dw_dt(x, p), in_axes=(0, None))(x, p)
        return jax.tree_util.tree_map(lambda a:jnp.mean(a, 0), q)

    return final_dw_dt

def sampler():
    return jax.random.uniform(KEY, (NUMSAMPLES, DIMS), jnp.float32, -1, 1)

In [ ]:
def foo(x, p):
    w = p[0].reshape(NUMPARAMS, DIMS)
    b = p[1]
    c = p[2]
    return c @ jnp.sin(w @ x + b)

In [ ]:
def make_x_grid(xmin=-1.0, xmax=1.0, ymin=-1.0, ymax=1.0, nx=50, ny=50):
    xs = jnp.linspace(xmin, xmax, nx)
    ys = jnp.linspace(ymin, ymax, ny)
    xx, yy = jnp.meshgrid(xs, ys, indexing="xy")
    return jnp.stack([xx, yy], axis=-1)

def foo_grid(p, x_grid):
    flat_x = x_grid.reshape(-1, DIMS)
    vals = jax.vmap(lambda x: foo(x, p))(flat_x)
    return vals.reshape(x_grid.shape[:-1])

In [ ]:
def make_p_evolution(p_evolution):
    """
    Преобразует эволюцию параметров из объекта Diffrax или кортежей 
    в плоский список состояний по шагам времени.
    """
    if hasattr(p_evolution, "ys"):
        p_evolution = [tuple(arr[i] for arr in p_evolution.ys) for i in range(p_evolution.ys[0].shape[0])]
    elif isinstance(p_evolution, tuple) and p_evolution and getattr(p_evolution[0], "ndim", 0) > 1:
        p_evolution = [tuple(arr[i] for arr in p_evolution) for i in range(p_evolution[0].shape[0])]
    return p_evolution

def plot_foo_timeline(x_grid, p_evolution, title="foo evolution"):
    """
    Строит анимированный квадратный тепловой график (Heatmap) в Plotly,
    отображающий эволюцию системы по шагам времени.
    """
    p_evolution = make_p_evolution(p_evolution)
    x = x_grid[0, :, 0].tolist()
    y = x_grid[:, 0, 1].tolist()

    # foo_grid должна быть определена в вашей области видимости
    z_list = [foo_grid(p, x_grid) for p in p_evolution]
    zmin = float(min(jnp.min(z) for z in z_list))
    zmax = float(max(jnp.max(z) for z in z_list))

    def heatmap(z):
        return go.Heatmap(
            z=z.tolist(),
            x=x,
            y=y,
            colorscale="Viridis",
            zmin=zmin,
            zmax=zmax,
            colorbar=dict(title="foo")
        )

    frames = [
        go.Frame(
            data=[heatmap(z)],
            name=str(i),
            layout=go.Layout(title_text=f"{title} — step {i}")
        )
        for i, z in enumerate(z_list)
    ]

    fig = go.Figure(
        data=[heatmap(z_list[0])],
        frames=frames,
        layout=go.Layout(
            title=title,
            # Фиксируем физические размеры контейнера (в пикселях)
            width=650,
            height=650,
            
            # Настраиваем оси так, чтобы график не растягивался
            xaxis=dict(
                title="x",
                scaleanchor="y",  # Масштаб оси X жестко привязывается к оси Y
                scaleratio=1      # Соотношение масштабов физических пикселей 1:1
            ),
            yaxis=dict(
                title="y"
            ),
            updatemenus=[
                dict(
                    type="buttons",
                    showactive=False,
                    y=1.1,
                    x=1.05,
                    xanchor="right",
                    yanchor="top",
                    buttons=[
                        dict(
                            label="Play",
                            method="animate",
                            args=[
                                None,
                                dict(
                                    frame=dict(duration=250, redraw=True),
                                    transition=dict(duration=0),
                                    fromcurrent=True,
                                    mode="immediate",
                                ),
                            ],
                        ),
                        dict(
                            label="Pause",
                            method="animate",
                            args=[
                                [None],
                                dict(frame=dict(duration=0, redraw=False), mode="immediate"),
                            ],
                        ),
                    ],
                )
            ],
            sliders=[
                dict(
                    active=0,
                    currentvalue=dict(prefix="step: "),
                    pad=dict(t=50),
                    steps=[
                        dict(
                            label=str(i),
                            method="animate",
                            args=[
                                [str(i)],
                                dict(frame=dict(duration=0, redraw=True), transition=dict(duration=0), mode="immediate"),
                            ],
                        )
                        for i in range(len(frames))
                    ],
                )
            ],
        ),
    )

    return fig

In [ ]:
dw_dt = get_dw_dt(foo, 0.01, sampler)
w0 = (
    jax.random.normal(KEY, (NUMPARAMS * DIMS))*5,
    jax.random.normal(KEY, (NUMPARAMS))*5,
    jax.random.normal(KEY, (NUMPARAMS))*5
)

In [ ]:
solver = diffrax.Dopri5()
term = diffrax.ODETerm(lambda t, y, args: dw_dt(y))
saveat = diffrax.SaveAt(ts=jnp.linspace(0, 10, 50))
sol = diffrax.diffeqsolve(term, solver, t0=0, t1=10, dt0=0.01, y0=w0, saveat=saveat)

In [ ]:
x_grid = make_x_grid()
plot_foo_timeline(x_grid, sol)